In [ ]:
from search_utils.utils import define_env
from search_utils.SearchAlgorithms import LowestSearch_Poda, HighestSearch_Poda, DFS_secundario
from adabyron_environment.AdaByronDEMO_v2 import AdaByronDEMO
from adabyron_environment.mercadoSecundario_v2 import MercadoSecundario
from adabyron_environment.agents.storageBattery_v2 import StorageBattery
from adabyron_environment.agents.storageBattery_mercado_secundario import StorageBatteryMS
from adabyron_environment.agents.airConditioning_v2 import Airconditioning
from queue import PriorityQueue
from itertools import permutations
from datetime import date

import random
import itertools
import pandas as pd
import matplotlib.pyplot as plt
import json
import datetime

In [ ]:
bandaSecundaria = pd.read_csv('test_data/bandaSecundaria.csv', sep=';')
bandaSecundaria['value'] = bandaSecundaria['value']
bandaSecundaria = list(bandaSecundaria['value'])

regulacionSubir = pd.read_csv('test_data/regulacionSubir.csv', sep=';')
regulacionSubir['value'] = regulacionSubir['value']
regulacionSubir = list(regulacionSubir['value'])

regulacionBajar = pd.read_csv('test_data/regulacionBajar.csv', sep=';')
regulacionBajar['value'] = regulacionBajar['value']
regulacionBajar = list(regulacionBajar['value'])

In [ ]:
highPercentage  = [0.00, 57.23, 99.90, 79.97, 41.21, 9.35, 0.00,  0.00, 41.78, 24.21, 49.70, 33.65, 9.66, 1.97, 25.84, 83.23, 0.00,  0.00,  0.00,  0.00,  0.00,  0.00,  0.00,  0.00]
lowPercentage   = [9.03, 0.00,  0.00,  0.00,  0.00,  0.00, 60.97, 0.96, 0.00,  0.00,  0.00,  0.00,  0.00, 0.00, 0.00,  0.00,  65.25, 24.78, 61.28, 65.14, 72.64, 99.30, 99.36, 69.52]



mercadoSecundario = MercadoSecundario(preservedEnergy=0.50, 
                                      priceCurve= bandaSecundaria, 
                                      highCurve= regulacionSubir, 
                                      lowCurve= regulacionBajar,
                                      highPercentage= highPercentage,
                                      lowPercentage= lowPercentage,
                                      battery= StorageBatteryMS("B1",capacity= 36, original_soc = 23))

# Con 25 resultados tarda 8m (original soc = 75)
first_state = mercadoSecundario.reset()
print(first_state)
resultado_mercado_secundario = DFS_secundario(first_state, mercadoSecundario, 30)

In [ ]:
arrayGrande = []
for r in resultado_mercado_secundario:
    res = []
    actual = r
    while(True):
        res.append(actual.cumulative_price)
        if actual.day_step == 0:
            break
        actual = actual.father
    res.reverse()
    arrayGrande.append(res)

In [ ]:
for a in arrayGrande:
    plt.plot(a)

plt.savefig("images\\Search\\MercadoSecundario\\DFS\\Balance_Ganado_{}.png".format(date.today()))

In [ ]:
best_solution = resultado_mercado_secundario[0]
for solution in resultado_mercado_secundario:
    if best_solution.cumulative_price < solution.cumulative_price:
        best_solution = solution

In [ ]:
actual = best_solution
soc = []
estado = []
while(True):

    soc.append(actual.agents_info["B1"]["soc"])
    estado.append(actual)
    if actual.day_step == 0:
        break
    actual = actual.father
estado.reverse()
soc.reverse()

In [ ]:
for i in estado:
    print(i)

In [ ]:
plt.plot(soc)
plt.show()

In [ ]:
# adaByron = {
#     "Facility": "adabyron",
#     "ExpectedProfit": estado[len(estado)-1].cumulative_price,
#     "states" :  [
#         {
#             "TimeStamp" : datetime.time(int(i.day_step/4), i.day_step%4*15),
#             "actions" : i.actions
#         } for i in estado
#                 ]
# }